In [1]:
import gsd.hoomd
from scipy.spatial import ConvexHull
import numpy as np 
import freud

import numpy as np
import matplotlib.pyplot as plt
import alphashape
from shapely.geometry import Polygon

import signac
project=signac.get_project()

In [2]:
def compute_3D_IQ(N_part, N_mesh, frame): 
    result = ""
    for i in range(N_part): 
        start = i * N_mesh
        end = (i + 1) * N_mesh
        mesh_positions = frame.particles.position[start:end]
        pts = box.unwrap(mesh_positions,imgs = frame.particles.image[start:end])
        hull = ConvexHull(pts)
        volume = hull.volume
        area = hull.area
        IQ = 36 * np.pi * volume **2 / area**3
        result += f"mesh {i} : {IQ:.3f}\n"
    return result

def compute_2D_IQ(N_part, N_mesh, frame): 
    result = ""
    total = 0
    for i in range(N_part): 
        start = i * N_mesh
        end = (i + 1) * N_mesh
        mesh_positions = frame.particles.position[start:end]
        pts = box.unwrap(mesh_positions,imgs = frame.particles.image[start:end])
        hull = ConvexHull(pts[:, 0:2])
        volume = hull.volume
        area = hull.area
        IQ = 4 * np.pi * volume / area**2
        total += IQ
        result += f"mesh {i} : {IQ:.3f}\n"
        if i == N_part-1: 
            avg = total / N_part
            result += f"average IQ: {avg:.3f}"
    return result

In [9]:
def alpha_shape_IQ(frame, N_part, N_mesh, alpha = None):
    """
    computes the perimeter, area, and isoperimetric quotient of a 2D alpha shape (irregular shape)
    """
    result = ""
    total = 0
    for i in range(N_part): 
        start = i * N_mesh
        end = (i + 1) * N_mesh
        mesh_positions = frame.particles.position[start:end]
        pts=box.unwrap(mesh_positions,imgs = frame.particles.image[start:end])
        pts = pts[:, 0:2] 
        if alpha is None: 
            shape = alphashape.alphashape(pts)
        else: 
            shape = alphashape.alphashape(pts, alpha) 
        area = shape.area
        perimeter = shape.length
        IQ = 4 * np.pi * area / perimeter**2
        total += IQ
        result += f"mesh {i} : IQ:{IQ:.3f}, perimeter:{perimeter:.3f}, area:{area:.3f}\n"
        if i == N_part-1: 
            avg = total / N_part
            result += f"average IQ: {avg:.3f}"
        plt.figure(figsize=(4,4))
        plt.scatter(pts[:,0], 
                    pts[:,1], 
                    s=10)
        x,y = shape.exterior.xy
        plt.plot(x,y,linewidth=2)
        plt.axis("equal")
        plt.xlabel("x")
        plt.ylabel("y")
        plt.title(f"mesh {i}", fontsize = 16)
        plt.show
    return result


def packing_eff(frame, N_part, N_mesh, alpha = None): 
    total_area = 0
    Lx = frame.configuration.box[0]
    Ly = frame.configuration.box[1]
    for i in range(N_part): 
        start = i * N_mesh
        end = (i + 1) * N_mesh
        mesh_positions = frame.particles.position[start:end]
        pts=box.unwrap(mesh_positions,imgs = frame.particles.image[start:end])
        pts = pts[:, 0:2] 
        if alpha is None: 
            shape = alphashape.alphashape(pts)
        else: 
            shape = alphashape.alphashape(pts, alpha) 
        area = shape.area
        total_area += area
    box_area = Lx * Ly
    efficiency = total_area / box_area
    return float(f"{efficiency:.3g}")

In [ ]:
for job in project:
    if job.isfile('trajectory.gsd') and job.id == "89433309306eb7d9b0dee764e4d8d6d8": 
        traj = gsd.hoomd.open(job.fn("trajectory.gsd")) 
        break
print("total number of frames: ", len(traj))
frame = traj[-1]
box = freud.box.Box.from_box(frame.configuration.box)
# print("2D original isoperimetric quotients:\n"+compute_2D_IQ(job.sp.N_part, job.sp.N_mesh, frame))
print("2D alpha shape:\n"+alpha_shape_IQ(frame, job.sp.N_part, job.sp.N_mesh, alpha = 0.7))
# print("Packing efficiency: ", packing_eff(frame, job.sp.N_part, job.sp.N_mesh, alpha = 0.7))